# PUMA Segmentation - Training on Colab Pro (G4 - 95.6 GB VRAM)

**Auto-resume**: Checkpoints saved to Google Drive every epoch.  
If Colab disconnects, just **Run All** again - training resumes automatically.  
Config: `configs/colab_g4.yaml`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify dataset on Drive
%ls /content/drive/MyDrive/dataset_PUMA/

In [ ]:
import os

REPO_DIR = "/content/drive/MyDrive/Segment_PUMA"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/hoangtung386/Segment_PUMA.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

In [ ]:
# Install PyTorch with Blackwell (sm_120) support, then other dependencies
!pip install -qqq torch torchvision --index-url https://download.pytorch.org/whl/cu126
!pip install -qqq -r {REPO_DIR}/requirements.txt
!pip install -qqq mamba-ssm

In [ ]:
# Login W&B to monitor training
import wandb
from google.colab import userdata

wandb.login(key=userdata.get('WANDB_API_KEY'))

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# (Optional) Run BEFORE training to prevent idle disconnect
import IPython
from google.colab import output

display(IPython.display.Javascript('''
function KeepAlive() {
    console.log("Keeping alive " + new Date().toLocaleString());
    google.colab.kernel.invokeFunction("notebook.ping", [], {});
}
setInterval(KeepAlive, 5 * 60 * 1000);
'''))
output.register_callback('notebook.ping', lambda: print('.', end='', flush=True))
print("Keep-alive enabled (ping every 5 min)")

In [ ]:
# Train
%cd /content/drive/MyDrive/Segment_PUMA
!python scripts/train.py --devices 0 --config configs/colab_g4.yaml

In [ ]:
# Evaluate best model: metrics + confusion matrix + prediction visualizations
%cd /content/drive/MyDrive/Segment_PUMA
!python scripts/evaluate.py \
    --checkpoint /content/drive/MyDrive/Segment_PUMA/checkpoints/best_puma.pth \
    --config configs/colab_g4.yaml \
    --split test \
    --devices 0 \
    --num-samples 5

## Display evaluation results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob

eval_dir = "/content/drive/MyDrive/Segment_PUMA/outputs/evaluation"

# Show report
report_path = f"{eval_dir}/evaluation_report.txt"
if os.path.exists(report_path):
    with open(report_path) as f:
        print(f.read())

In [ ]:
# Show confusion matrix
cm_path = f"{eval_dir}/confusion_matrix.png"
if os.path.exists(cm_path):
    plt.figure(figsize=(12, 5))
    plt.imshow(mpimg.imread(cm_path))
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.show()

In [ ]:
# Show prediction visualizations
pred_files = sorted(glob.glob(f"{eval_dir}/prediction_*.png"))
for img_path in pred_files:
    plt.figure(figsize=(15, 5))
    plt.imshow(mpimg.imread(img_path))
    plt.axis('off')
    plt.title(os.path.basename(img_path))
    plt.show()

## Training complete

In [ ]:
# Disconnect runtime to stop billing
from google.colab import runtime
print("Training & evaluation finished! Disconnecting runtime...")
runtime.unassign()